
# STEP 5 — Reviewer-Killer Robustness & Ablations

This notebook is designed to attack the easiest reviewer objections **before** paper writing.

It reuses outputs already generated in Steps 4A–4C and focuses on:

1. **Second-model seed robustness** — Flowers-64, \(C=100\) vs \(C=200\)
2. **Training-data robustness** — CIFAR-10, \(C=50\) vs \(C=500\)
3. **Architecture robustness** — Local CNN vs Dilated CNN vs small ResNet corrector
4. **Capacity robustness / saturation** — does the separation disappear with more parameters?
5. **Metric robustness** — NMSE + normalized MAE + residual cosine similarity
6. **Equal-NFE solver inversion audit** — same diffusion-model evaluation budget
7. **Threshold/RAC sensitivity** — avoid cherry-picking a single \(\rho\)

The notebook does **not** regenerate expensive diffusion trajectories.  
It expects the saved `.pt` and `.csv` files from previous steps.


In [1]:

from pathlib import Path
import copy
import math
import gc
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED_BASE = 20260925

random.seed(SEED_BASE)
np.random.seed(SEED_BASE)
torch.manual_seed(SEED_BASE)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce RTX 2060


## 0. Paths and required files

In [2]:

ROOT = Path(".")

CIFAR_DIR = ROOT / "ddim_residual_data_512"
SOLVER_DIR = ROOT / "solver_generalization"
FLOWERS_DIR = ROOT / "flowers64_step4c"
OUT_DIR = ROOT / "step5_robustness"

OUT_DIR.mkdir(exist_ok=True)

required = [
    CIFAR_DIR / "z.pt",
    CIFAR_DIR / "teacher_1000.pt",
    CIFAR_DIR / "baseline_50.pt",
    CIFAR_DIR / "baseline_500.pt",
    FLOWERS_DIR / "z.pt",
    FLOWERS_DIR / "teacher_ddim_500.pt",
    FLOWERS_DIR / "baseline_ddim_100.pt",
    FLOWERS_DIR / "baseline_ddim_200.pt",
]

for p in required:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing required file: {p.resolve()}\n"
            "Run the previous experiment notebooks first."
        )

print("Core saved tensors found.")

# Optional equal-NFE files
optional_csvs = [
    SOLVER_DIR / "equal_nfe_residual_magnitude.csv",
    SOLVER_DIR / "reference_sensitivity_learning.csv",
]

for p in optional_csvs:
    print(p, "->", p.exists())


Core saved tensors found.
solver_generalization\equal_nfe_residual_magnitude.csv -> True
solver_generalization\reference_sensitivity_learning.csv -> True


## 1. Shared correction architectures

In [3]:

class LocalCorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, 3, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


class DilatedCorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1, dilation=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=2, dilation=2),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=4, dilation=4),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1, dilation=1),
            nn.GELU(),
            nn.Conv2d(width, 3, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(width, width, 3, padding=1),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.block(x))


class ResCorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.in_proj = nn.Conv2d(3, width, 3, padding=1)
        self.body = nn.Sequential(
            ResidualBlock(width),
            ResidualBlock(width),
            ResidualBlock(width),
        )
        self.out_proj = nn.Conv2d(width, 3, 3, padding=1)

    def forward(self, x):
        x = self.in_proj(x)
        x = self.body(x)
        return self.out_proj(x)


ARCH_FACTORIES = {
    "local": LocalCorrectionCNN,
    "dilated": DilatedCorrectionCNN,
    "resnet": ResCorrectionCNN,
}


def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


for arch, factory in ARCH_FACTORIES.items():
    print("\n", arch)
    for width in [4, 8, 16, 32]:
        m = factory(width)
        print(
            f"width={width:2d} params={count_params(m):7d}"
        )



 local
width= 4 params=    519
width= 8 params=   1611
width=16 params=   5523
width=32 params=  20259

 dilated
width= 4 params=    667
width= 8 params=   2195
width=16 params=   7843
width=32 params=  29507

 resnet
width= 4 params=   1111
width= 8 params=   3947
width=16 params=  14803
width=32 params=  57251



## 2. Shared robust trainer

Important distinctions:
- target normalization uses **train residual only**
- model selection uses **validation NMSE only**
- final reporting uses held-out **test**
- multiple metrics are returned:
  - NMSE
  - normalized MAE
  - residual cosine similarity

The theory is based on \(L^2\)/NMSE; the other metrics are empirical robustness checks.


In [4]:

def _metrics(pred, target):
    mse = ((pred - target) ** 2).mean().item()
    energy = (target ** 2).mean().item()
    nmse = mse / energy

    mae = (pred - target).abs().mean().item()
    target_abs = target.abs().mean().item()
    nmae = mae / target_abs

    p = pred.flatten(1)
    t = target.flatten(1)

    cosine = torch.nn.functional.cosine_similarity(
        p,
        t,
        dim=1,
        eps=1e-8
    ).mean().item()

    return {
        "nmse": nmse,
        "nmae": nmae,
        "cosine": cosine,
    }


def train_corrector(
    z_train,
    r_train,
    z_val,
    r_val,
    z_test,
    r_test,
    width,
    architecture="local",
    max_epochs=800,
    batch_size=32,
    lr=3e-4,
    patience=60,
    seed=0,
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = ARCH_FACTORIES[architecture](width).to(device)

    for name, tensor in {
        "z_train": z_train,
        "r_train": r_train,
        "z_val": z_val,
        "r_val": r_val,
        "z_test": z_test,
        "r_test": r_test,
    }.items():
        if not torch.isfinite(tensor).all():
            raise ValueError(f"{name} contains NaN/Inf.")

    train_energy = (r_train ** 2).mean().item()
    if train_energy <= 1e-14:
        raise ValueError("Train residual energy is effectively zero.")

    train_rms = train_energy ** 0.5
    y_train = r_train / train_rms

    loader = DataLoader(
        TensorDataset(z_train, y_train),
        batch_size=batch_size,
        shuffle=True,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    loss_fn = nn.MSELoss()

    val_energy = (r_val ** 2).mean().item()

    model.eval()
    with torch.no_grad():
        pred_val = (
            model(z_val.to(device)).cpu()
            * train_rms
        )

    best_val_nmse = (
        ((pred_val - r_val) ** 2).mean().item()
        / val_energy
    )

    if not math.isfinite(best_val_nmse):
        raise ValueError("Initial validation NMSE is non-finite.")

    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    bad_epochs = 0

    for epoch in range(max_epochs):
        model.train()
        valid_epoch = True

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)

            if not torch.isfinite(loss):
                valid_epoch = False
                break

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

        if not valid_epoch:
            break

        model.eval()

        with torch.no_grad():
            pred_val = (
                model(z_val.to(device)).cpu()
                * train_rms
            )

        val_nmse = (
            ((pred_val - r_val) ** 2).mean().item()
            / val_energy
        )

        if not math.isfinite(val_nmse):
            break

        if val_nmse < best_val_nmse - 1e-5:
            best_val_nmse = val_nmse
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_train = (
            model(z_train.to(device)).cpu()
            * train_rms
        )
        pred_test = (
            model(z_test.to(device)).cpu()
            * train_rms
        )

    train_metrics = _metrics(pred_train, r_train)
    test_metrics = _metrics(pred_test, r_test)

    result = {
        "architecture": architecture,
        "width": width,
        "params": count_params(model),
        "seed": seed,
        "best_epoch": best_epoch,
        "train_rms": train_rms,
        "train_nmse": train_metrics["nmse"],
        "test_nmse": test_metrics["nmse"],
        "test_nmae": test_metrics["nmae"],
        "test_cosine": test_metrics["cosine"],
        "val_nmse": best_val_nmse,
    }

    del model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return result


print("Shared trainer ready.")


Shared trainer ready.



# 5A — Flowers-64: 5-seed external-replication robustness

This is the most important immediate robustness test because the observed
\(C=100 \rightarrow 200\) inversion was from one training seed.

We test:
- \(C\in\{100,200\}\)
- widths \(4,16\)
- 5 seeds
- same train/val/test split as Step 4C

A strong result is:
\[
\operatorname{mean}_s E_{200,s}(K) >
\operatorname{mean}_s E_{100,s}(K)
\]
for both tested capacities, with small/moderate seed variance.


In [5]:

flowers_z = torch.load(
    FLOWERS_DIR / "z.pt",
    map_location="cpu"
)

flowers_teacher = torch.load(
    FLOWERS_DIR / "teacher_ddim_500.pt",
    map_location="cpu"
)

flowers_baselines = {
    100: torch.load(
        FLOWERS_DIR / "baseline_ddim_100.pt",
        map_location="cpu"
    ),
    200: torch.load(
        FLOWERS_DIR / "baseline_ddim_200.pt",
        map_location="cpu"
    ),
}

F_TRAIN = 192
F_VAL_END = 224

fz_train = flowers_z[:F_TRAIN]
fz_val = flowers_z[F_TRAIN:F_VAL_END]
fz_test = flowers_z[F_VAL_END:]

FLOWERS_SEEDS = [0, 1, 2, 3, 4]
FLOWERS_WIDTHS = [4, 16]

flowers_seed_path = (
    OUT_DIR / "flowers_seed_robustness.csv"
)

if flowers_seed_path.exists():
    flowers_seed_results = (
        pd.read_csv(flowers_seed_path)
        .to_dict("records")
    )
else:
    flowers_seed_results = []


def flowers_done(C, width, seed):
    return any(
        int(r["C"]) == C
        and int(r["width"]) == width
        and int(r["seed"]) == seed
        for r in flowers_seed_results
    )


for C in [100, 200]:
    R = flowers_teacher - flowers_baselines[C]

    r_train = R[:F_TRAIN]
    r_val = R[F_TRAIN:F_VAL_END]
    r_test = R[F_VAL_END:]

    for width in FLOWERS_WIDTHS:
        for seed in FLOWERS_SEEDS:

            if flowers_done(C, width, seed):
                print(
                    "SKIP",
                    C,
                    width,
                    seed
                )
                continue

            print(
                "\nFLOWERS",
                "C=", C,
                "width=", width,
                "seed=", seed
            )

            out = train_corrector(
                fz_train,
                r_train,
                fz_val,
                r_val,
                fz_test,
                r_test,
                width=width,
                architecture="local",
                max_epochs=1000,
                batch_size=16,
                patience=80,
                seed=seed,
            )

            out["C"] = C
            flowers_seed_results.append(out)

            pd.DataFrame(
                flowers_seed_results
            ).to_csv(
                flowers_seed_path,
                index=False
            )

flowers_seed_df = pd.DataFrame(
    flowers_seed_results
)

flowers_seed_summary = (
    flowers_seed_df
    .groupby(["C", "width", "params"], as_index=False)
    .agg(
        nmse_mean=("test_nmse", "mean"),
        nmse_std=("test_nmse", "std"),
        nmae_mean=("test_nmae", "mean"),
        cosine_mean=("test_cosine", "mean"),
        train_nmse_mean=("train_nmse", "mean"),
        best_epoch_mean=("best_epoch", "mean"),
    )
)

display(flowers_seed_summary)


C:\Users\User\AppData\Local\Temp\ipykernel_5060\3112818366.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  flowers_z = torch.load(
C:\Users\User\AppData\Local\Temp\ipyke


FLOWERS C= 100 width= 4 seed= 0

FLOWERS C= 100 width= 4 seed= 1

FLOWERS C= 100 width= 4 seed= 2

FLOWERS C= 100 width= 4 seed= 3

FLOWERS C= 100 width= 4 seed= 4

FLOWERS C= 100 width= 16 seed= 0

FLOWERS C= 100 width= 16 seed= 1

FLOWERS C= 100 width= 16 seed= 2

FLOWERS C= 100 width= 16 seed= 3

FLOWERS C= 100 width= 16 seed= 4

FLOWERS C= 200 width= 4 seed= 0

FLOWERS C= 200 width= 4 seed= 1

FLOWERS C= 200 width= 4 seed= 2

FLOWERS C= 200 width= 4 seed= 3

FLOWERS C= 200 width= 4 seed= 4

FLOWERS C= 200 width= 16 seed= 0

FLOWERS C= 200 width= 16 seed= 1

FLOWERS C= 200 width= 16 seed= 2

FLOWERS C= 200 width= 16 seed= 3

FLOWERS C= 200 width= 16 seed= 4


,C,width,params,nmse_mean,nmse_std,nmae_mean,cosine_mean,train_nmse_mean,best_epoch_mean
0,100,4,519,0.782636,0.005646,0.887407,0.470986,0.781643,419.8
1,100,16,5523,0.776828,0.006311,0.887375,0.477885,0.765444,342.8
2,200,4,519,0.793842,0.002785,0.889041,0.468215,0.785873,579.4
3,200,16,5523,0.775931,0.003632,0.878094,0.490399,0.760315,509.2


In [6]:

# Paired seed differences: C=200 minus C=100
flowers_pairs = []

for width in FLOWERS_WIDTHS:
    a = (
        flowers_seed_df[
            (flowers_seed_df["C"] == 100)
            & (flowers_seed_df["width"] == width)
        ]
        .set_index("seed")
    )

    b = (
        flowers_seed_df[
            (flowers_seed_df["C"] == 200)
            & (flowers_seed_df["width"] == width)
        ]
        .set_index("seed")
    )

    common = sorted(
        set(a.index).intersection(b.index)
    )

    diffs = (
        b.loc[common, "test_nmse"].values
        - a.loc[common, "test_nmse"].values
    )

    flowers_pairs.append({
        "width": width,
        "n_seeds": len(common),
        "mean_delta_nmse_200_minus_100":
            float(np.mean(diffs)),
        "std_delta":
            float(np.std(diffs, ddof=1))
            if len(diffs) > 1 else np.nan,
        "fraction_positive":
            float(np.mean(diffs > 0)),
        "all_positive":
            bool(np.all(diffs > 0)),
    })

flowers_pair_df = pd.DataFrame(
    flowers_pairs
)

display(flowers_pair_df)

flowers_pair_df.to_csv(
    OUT_DIR / "flowers_paired_inversion.csv",
    index=False
)


,width,n_seeds,mean_delta_nmse_200_minus_100,std_delta,fraction_positive,all_positive
0,4,5,0.011206,0.003992,1.0,True
1,16,5,-0.000897,0.004352,0.4,False



# 5B — CIFAR: training-data robustness

Reviewer objection:
> “Maybe \(R_{500}\) only looks harder because the corrector is data-limited.”

We keep:
- fixed validation = 64
- fixed test = 64
- same held-out examples
- width = 8
- 3 seeds
- train sizes \(96,192,384\)

We compare \(C=50\) vs \(C=500\).


In [7]:

cifar_z = torch.load(
    CIFAR_DIR / "z.pt",
    map_location="cpu"
)

cifar_teacher = torch.load(
    CIFAR_DIR / "teacher_1000.pt",
    map_location="cpu"
)

cifar_baselines = {
    50: torch.load(
        CIFAR_DIR / "baseline_50.pt",
        map_location="cpu"
    ),
    500: torch.load(
        CIFAR_DIR / "baseline_500.pt",
        map_location="cpu"
    ),
}

# Same original held-out sets
cz_val = cifar_z[384:448]
cz_test = cifar_z[448:512]

TRAIN_SIZES = [96, 192, 384]
DATA_SEEDS = [0, 1, 2]
DATA_WIDTH = 8

data_path = (
    OUT_DIR / "cifar_data_size_robustness.csv"
)

if data_path.exists():
    data_results = (
        pd.read_csv(data_path)
        .to_dict("records")
    )
else:
    data_results = []


def data_done(C, n_train, seed):
    return any(
        int(r["C"]) == C
        and int(r["n_train"]) == n_train
        and int(r["seed"]) == seed
        for r in data_results
    )


for C in [50, 500]:
    R = cifar_teacher - cifar_baselines[C]

    r_val = R[384:448]
    r_test = R[448:512]

    for n_train in TRAIN_SIZES:
        z_train = cifar_z[:n_train]
        r_train = R[:n_train]

        for seed in DATA_SEEDS:
            if data_done(C, n_train, seed):
                print(
                    "SKIP",
                    C,
                    n_train,
                    seed
                )
                continue

            print(
                "\nCIFAR DATA",
                "C=", C,
                "N=", n_train,
                "seed=", seed
            )

            out = train_corrector(
                z_train,
                r_train,
                cz_val,
                r_val,
                cz_test,
                r_test,
                width=DATA_WIDTH,
                architecture="local",
                max_epochs=1000,
                batch_size=32,
                patience=60,
                seed=seed,
            )

            out["C"] = C
            out["n_train"] = n_train

            data_results.append(out)

            pd.DataFrame(
                data_results
            ).to_csv(
                data_path,
                index=False
            )

data_df = pd.DataFrame(data_results)

data_summary = (
    data_df
    .groupby(["C", "n_train"], as_index=False)
    .agg(
        nmse_mean=("test_nmse", "mean"),
        nmse_std=("test_nmse", "std"),
        nmae_mean=("test_nmae", "mean"),
        cosine_mean=("test_cosine", "mean"),
    )
)

display(data_summary)


C:\Users\User\AppData\Local\Temp\ipykernel_5060\1887418836.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cifar_z = torch.load(
C:\Users\User\AppData\Local\Temp\ipykern


CIFAR DATA C= 50 N= 96 seed= 0

CIFAR DATA C= 50 N= 96 seed= 1

CIFAR DATA C= 50 N= 96 seed= 2

CIFAR DATA C= 50 N= 192 seed= 0

CIFAR DATA C= 50 N= 192 seed= 1

CIFAR DATA C= 50 N= 192 seed= 2

CIFAR DATA C= 50 N= 384 seed= 0

CIFAR DATA C= 50 N= 384 seed= 1

CIFAR DATA C= 50 N= 384 seed= 2

CIFAR DATA C= 500 N= 96 seed= 0

CIFAR DATA C= 500 N= 96 seed= 1

CIFAR DATA C= 500 N= 96 seed= 2

CIFAR DATA C= 500 N= 192 seed= 0

CIFAR DATA C= 500 N= 192 seed= 1

CIFAR DATA C= 500 N= 192 seed= 2

CIFAR DATA C= 500 N= 384 seed= 0

CIFAR DATA C= 500 N= 384 seed= 1

CIFAR DATA C= 500 N= 384 seed= 2


,C,n_train,nmse_mean,nmse_std,nmae_mean,cosine_mean
0,50,96,0.329442,0.001958,0.552060,0.861827
1,50,192,0.328217,0.001074,0.552580,0.864823
2,50,384,0.321530,0.002262,0.543305,0.866857
3,500,96,0.559254,0.000690,0.679618,0.763652
4,500,192,0.556512,0.000377,0.676890,0.767324
5,500,384,0.551832,0.001397,0.673886,0.768063


In [8]:

data_pivot = data_summary.pivot(
    index="n_train",
    columns="C",
    values="nmse_mean"
)

data_pivot["delta_500_minus_50"] = (
    data_pivot[500] - data_pivot[50]
)

display(data_pivot)

data_pivot.to_csv(
    OUT_DIR / "cifar_data_size_gap.csv"
)


C,50,500,delta_500_minus_50
n_train,,,
96,0.329442,0.559254,0.229812
192,0.328217,0.556512,0.228296
384,0.321530,0.551832,0.230302



# 5C — CIFAR: architecture robustness

Reviewer objection:
> “The effect is an artifact of one local CNN architecture.”

We compare three qualitatively different correctors:
- local CNN
- dilated CNN
- residual CNN

We use:
- \(C=50\) and \(C=500\)
- width 16
- 3 seeds

The parameter counts differ somewhat, so the main claim here is **ordering robustness within each architecture**, not exact cross-architecture fairness.


In [9]:

ARCH_SEEDS = [0, 1, 2]
ARCH_WIDTH = 16

arch_path = (
    OUT_DIR / "cifar_architecture_robustness.csv"
)

if arch_path.exists():
    arch_results = (
        pd.read_csv(arch_path)
        .to_dict("records")
    )
else:
    arch_results = []


def arch_done(C, architecture, seed):
    return any(
        int(r["C"]) == C
        and r["architecture"] == architecture
        and int(r["seed"]) == seed
        for r in arch_results
    )


cz_train = cifar_z[:384]

for C in [50, 500]:
    R = cifar_teacher - cifar_baselines[C]

    r_train = R[:384]
    r_val = R[384:448]
    r_test = R[448:512]

    for architecture in [
        "local",
        "dilated",
        "resnet"
    ]:
        for seed in ARCH_SEEDS:

            if arch_done(
                C,
                architecture,
                seed
            ):
                print(
                    "SKIP",
                    C,
                    architecture,
                    seed
                )
                continue

            print(
                "\nARCH",
                "C=", C,
                architecture,
                "seed=", seed
            )

            out = train_corrector(
                cz_train,
                r_train,
                cz_val,
                r_val,
                cz_test,
                r_test,
                width=ARCH_WIDTH,
                architecture=architecture,
                max_epochs=1000,
                batch_size=32,
                patience=60,
                seed=seed,
            )

            out["C"] = C
            arch_results.append(out)

            pd.DataFrame(
                arch_results
            ).to_csv(
                arch_path,
                index=False
            )

arch_df = pd.DataFrame(arch_results)

arch_summary = (
    arch_df
    .groupby(
        ["architecture", "C", "params"],
        as_index=False
    )
    .agg(
        nmse_mean=("test_nmse", "mean"),
        nmse_std=("test_nmse", "std"),
        nmae_mean=("test_nmae", "mean"),
        cosine_mean=("test_cosine", "mean"),
    )
)

display(arch_summary)



ARCH C= 50 local seed= 0

ARCH C= 50 local seed= 1

ARCH C= 50 local seed= 2

ARCH C= 50 dilated seed= 0

ARCH C= 50 dilated seed= 1

ARCH C= 50 dilated seed= 2

ARCH C= 50 resnet seed= 0

ARCH C= 50 resnet seed= 1

ARCH C= 50 resnet seed= 2

ARCH C= 500 local seed= 0

ARCH C= 500 local seed= 1

ARCH C= 500 local seed= 2

ARCH C= 500 dilated seed= 0

ARCH C= 500 dilated seed= 1

ARCH C= 500 dilated seed= 2

ARCH C= 500 resnet seed= 0

ARCH C= 500 resnet seed= 1

ARCH C= 500 resnet seed= 2


,architecture,C,params,nmse_mean,nmse_std,nmae_mean,cosine_mean
0,dilated,50,7843,0.298916,0.000950,0.520787,0.867466
1,dilated,500,7843,0.533599,0.000715,0.654141,0.769753
2,local,50,5523,0.320659,0.001146,0.544438,0.867463
3,local,500,5523,0.552337,0.000376,0.673234,0.767644
4,resnet,50,14803,0.309351,0.001171,0.533370,0.866044
5,resnet,500,14803,0.544975,0.001395,0.666232,0.766147


In [10]:

arch_gap = (
    arch_summary
    .pivot_table(
        index=["architecture"],
        columns="C",
        values="nmse_mean"
    )
)

arch_gap["delta_500_minus_50"] = (
    arch_gap[500] - arch_gap[50]
)

display(arch_gap)

arch_gap.to_csv(
    OUT_DIR / "cifar_architecture_gap.csv"
)


C,50,500,delta_500_minus_50
architecture,,,
dilated,0.298916,0.533599,0.234683
local,0.320659,0.552337,0.231678
resnet,0.309351,0.544975,0.235625



# 5D — CIFAR: capacity robustness

Reviewer objection:
> “\(R_{500}\) only looks harder because your correctors are too small.”

We sweep:
- widths \(8, 32, 128\)
- \(C=50\) vs \(500\)
- 3 seeds
- local architecture

If the normalized gap survives at width 128, the simple “just add capacity” explanation is weakened substantially.


In [11]:

CAP_WIDTHS = [8, 32, 128]
CAP_SEEDS = [0, 1, 2]

cap_path = (
    OUT_DIR / "cifar_capacity_robustness.csv"
)

if cap_path.exists():
    cap_results = (
        pd.read_csv(cap_path)
        .to_dict("records")
    )
else:
    cap_results = []


def cap_done(C, width, seed):
    return any(
        int(r["C"]) == C
        and int(r["width"]) == width
        and int(r["seed"]) == seed
        for r in cap_results
    )


for C in [50, 500]:
    R = cifar_teacher - cifar_baselines[C]

    r_train = R[:384]
    r_val = R[384:448]
    r_test = R[448:512]

    for width in CAP_WIDTHS:
        for seed in CAP_SEEDS:

            if cap_done(C, width, seed):
                print(
                    "SKIP",
                    C,
                    width,
                    seed
                )
                continue

            print(
                "\nCAPACITY",
                "C=", C,
                "width=", width,
                "seed=", seed
            )

            out = train_corrector(
                cz_train,
                r_train,
                cz_val,
                r_val,
                cz_test,
                r_test,
                width=width,
                architecture="local",
                max_epochs=1000,
                batch_size=32,
                patience=60,
                seed=seed,
            )

            out["C"] = C
            cap_results.append(out)

            pd.DataFrame(
                cap_results
            ).to_csv(
                cap_path,
                index=False
            )

cap_df = pd.DataFrame(cap_results)

cap_summary = (
    cap_df
    .groupby(
        ["C", "width", "params"],
        as_index=False
    )
    .agg(
        nmse_mean=("test_nmse", "mean"),
        nmse_std=("test_nmse", "std"),
        nmae_mean=("test_nmae", "mean"),
        cosine_mean=("test_cosine", "mean"),
    )
)

display(cap_summary)



CAPACITY C= 50 width= 8 seed= 0

CAPACITY C= 50 width= 8 seed= 1

CAPACITY C= 50 width= 8 seed= 2

CAPACITY C= 50 width= 32 seed= 0

CAPACITY C= 50 width= 32 seed= 1

CAPACITY C= 50 width= 32 seed= 2

CAPACITY C= 50 width= 128 seed= 0

CAPACITY C= 50 width= 128 seed= 1

CAPACITY C= 50 width= 128 seed= 2

CAPACITY C= 500 width= 8 seed= 0

CAPACITY C= 500 width= 8 seed= 1

CAPACITY C= 500 width= 8 seed= 2

CAPACITY C= 500 width= 32 seed= 0

CAPACITY C= 500 width= 32 seed= 1

CAPACITY C= 500 width= 32 seed= 2

CAPACITY C= 500 width= 128 seed= 0

CAPACITY C= 500 width= 128 seed= 1

CAPACITY C= 500 width= 128 seed= 2


,C,width,params,nmse_mean,nmse_std,nmae_mean,cosine_mean
0,50,8,1611,0.321530,0.002262,0.543305,0.866857
1,50,32,20259,0.318622,0.000969,0.542454,0.868285
2,50,128,302211,0.318836,0.001352,0.541119,0.870186
3,500,8,1611,0.551832,0.001397,0.673886,0.768063
4,500,32,20259,0.550978,0.001189,0.672407,0.770116
5,500,128,302211,0.551711,0.000664,0.673983,0.771663


In [12]:

cap_gap = cap_summary.pivot_table(
    index=["width", "params"],
    columns="C",
    values="nmse_mean"
)

cap_gap["delta_500_minus_50"] = (
    cap_gap[500] - cap_gap[50]
)

display(cap_gap)

cap_gap.to_csv(
    OUT_DIR / "cifar_capacity_gap.csv"
)


,C,50,500,delta_500_minus_50
width,params,,,
8,1611,0.321530,0.551832,0.230302
32,20259,0.318622,0.550978,0.232356
128,302211,0.318836,0.551711,0.232876



# 5E — Equal-NFE solver inversion audit

This uses Step 4A/4B saved results.

For the DDIM-1000 target, we ask at the **same NFE**:
- does lower residual MSE imply lower correction NMSE?
- are there actual same-compute accuracy–difficulty inversions?

This is especially valuable because it addresses the objection that different baseline accuracies merely reflect different compute budgets.


In [13]:

mag_path = (
    SOLVER_DIR / "equal_nfe_residual_magnitude.csv"
)

learn_path = (
    SOLVER_DIR / "reference_sensitivity_learning.csv"
)

if mag_path.exists() and learn_path.exists():

    solver_mag = pd.read_csv(mag_path)
    solver_learn = pd.read_csv(learn_path)

    # DDIM1000 is the primary target for this audit
    solver_learn = solver_learn[
        solver_learn["reference"] == "DDIM1000"
    ].copy()

    equal_nfe_rows = []

    for C in sorted(
        set(solver_learn["C"]).intersection(
            solver_mag["C"]
        )
    ):
        m = solver_mag[
            solver_mag["C"] == C
        ].set_index("solver")

        l = solver_learn[
            solver_learn["C"] == C
        ]

        for width in sorted(
            l["width"].unique()
        ):
            lw = (
                l[l["width"] == width]
                .set_index("solver")
            )

            common_solvers = sorted(
                set(m.index).intersection(
                    lw.index
                )
            )

            for i, s1 in enumerate(common_solvers):
                for s2 in common_solvers[i+1:]:

                    mse1 = float(
                        m.loc[s1, "residual_mse"]
                    )
                    mse2 = float(
                        m.loc[s2, "residual_mse"]
                    )

                    e1 = float(
                        lw.loc[s1, "test_nmse"]
                    )
                    e2 = float(
                        lw.loc[s2, "test_nmse"]
                    )

                    # record any reversed ordering
                    inversion = (
                        (mse1 < mse2 and e1 > e2)
                        or
                        (mse2 < mse1 and e2 > e1)
                    )

                    if inversion:
                        if mse1 < mse2:
                            more_accurate = s1
                            less_accurate = s2
                            nmse_more = e1
                            nmse_less = e2
                            mse_more = mse1
                            mse_less = mse2
                        else:
                            more_accurate = s2
                            less_accurate = s1
                            nmse_more = e2
                            nmse_less = e1
                            mse_more = mse2
                            mse_less = mse1

                        equal_nfe_rows.append({
                            "C": C,
                            "width": width,
                            "more_accurate_solver":
                                more_accurate,
                            "less_accurate_solver":
                                less_accurate,
                            "mse_more_accurate":
                                mse_more,
                            "mse_less_accurate":
                                mse_less,
                            "nmse_more_accurate":
                                nmse_more,
                            "nmse_less_accurate":
                                nmse_less,
                            "nmse_inversion_gap":
                                nmse_more - nmse_less,
                        })

    equal_nfe_inv = pd.DataFrame(
        equal_nfe_rows
    )

    if len(equal_nfe_inv):
        equal_nfe_inv = (
            equal_nfe_inv
            .sort_values(
                "nmse_inversion_gap",
                ascending=False
            )
            .reset_index(drop=True)
        )
        display(equal_nfe_inv)
    else:
        print(
            "No equal-NFE inversion found "
            "in the currently saved grid."
        )

    equal_nfe_inv.to_csv(
        OUT_DIR / "equal_nfe_inversions.csv",
        index=False
    )

else:
    print(
        "Equal-NFE saved CSVs not found. "
        "Skipping 5E."
    )


,C,width,more_accurate_solver,less_accurate_solver,mse_more_accurate,mse_less_accurate,nmse_more_accurate,nmse_less_accurate,nmse_inversion_gap
0,50,8,ddim,deis,0.215740,0.243225,0.363586,0.339413,0.024173
1,50,8,ddim,dpmpp,0.215740,0.243834,0.363586,0.340722,0.022864
2,50,8,ddim,unipc,0.215740,0.243029,0.363586,0.341390,0.022196
3,100,8,ddim,deis,0.196293,0.242124,0.361784,0.340362,0.021422
4,100,8,ddim,dpmpp,0.196293,0.242484,0.361784,0.340815,0.020969
5,100,4,ddim,dpmpp,0.196293,0.242484,0.362139,0.341209,0.020930
6,100,8,ddim,unipc,0.196293,0.242250,0.361784,0.341001,0.020783
7,100,4,ddim,unipc,0.196293,0.242250,0.362139,0.341403,0.020736
8,100,4,ddim,deis,0.196293,0.242124,0.362139,0.342786,0.019353
9,100,1,ddim,deis,0.196293,0.242124,0.411020,0.392354,0.018666



# 5F — RAC threshold sensitivity

A reviewer should not be able to say the conclusion depends on one hand-picked \(\rho\).

We build empirical RAC over a dense tolerance grid using the strongest available CIFAR capacity results.


In [14]:

rho_grid = np.round(
    np.arange(0.10, 0.91, 0.05),
    2
)

rac_rows = []

for rho in rho_grid:
    for C in [50, 500]:
        sub = (
            cap_df[cap_df["C"] == C]
            .groupby(["width", "params"], as_index=False)
            .agg(
                mean_nmse=("test_nmse", "mean"),
                std_nmse=("test_nmse", "std")
            )
            .sort_values("params")
        )

        feasible = sub[
            sub["mean_nmse"] <= rho
        ]

        if len(feasible) == 0:
            min_params = np.nan
            achieved = np.nan
        else:
            row = feasible.iloc[0]
            min_params = int(row["params"])
            achieved = float(
                row["mean_nmse"]
            )

        rac_rows.append({
            "rho": rho,
            "C": C,
            "min_params": min_params,
            "achieved_mean_nmse": achieved,
        })

rac_sensitivity = pd.DataFrame(
    rac_rows
)

rac_sensitivity_pivot = (
    rac_sensitivity
    .pivot(
        index="rho",
        columns="C",
        values="min_params"
    )
)

display(rac_sensitivity_pivot)

rac_sensitivity.to_csv(
    OUT_DIR / "rac_threshold_sensitivity.csv",
    index=False
)


C,50,500
rho,,
0.10,NaN,NaN
0.15,NaN,NaN
0.20,NaN,NaN
0.25,NaN,NaN
0.30,NaN,NaN
0.35,1611.0,NaN
0.40,1611.0,NaN
0.45,1611.0,NaN
0.50,1611.0,NaN



# 5G — Final reviewer-killer summary

Interpretation rules:
- **Flowers PASS** if the \(200-100\) paired NMSE gap is positive across most/all seeds.
- **Data-size PASS** if the \(500-50\) gap remains clearly positive as training data grows.
- **Architecture PASS** if \(500>50\) within all/most architecture families.
- **Capacity PASS** if \(500>50\) persists through the largest tested width.
- **Equal-NFE PASS** if at least one same-NFE solver pair exhibits reversed accuracy/difficulty ordering.
- **Metric robustness PASS** if NMSE ordering is accompanied by worse NMAE and/or lower cosine similarity in the same direction.


In [15]:

print("=" * 80)
print("STEP 5 SUMMARY")
print("=" * 80)

print("\n[5A] FLOWERS PAIRED SEED ROBUSTNESS")
display(flowers_pair_df)

print("\n[5B] CIFAR DATA-SIZE GAP")
display(data_pivot)

print("\n[5C] CIFAR ARCHITECTURE GAP")
display(arch_gap)

print("\n[5D] CIFAR CAPACITY GAP")
display(cap_gap)

print("\n[5E] EQUAL-NFE INVERSIONS")
if "equal_nfe_inv" in globals() and len(equal_nfe_inv):
    display(equal_nfe_inv.head(20))
else:
    print("No inversion table available.")

print("\n[5F] RAC THRESHOLD SENSITIVITY")
display(rac_sensitivity_pivot)

# Compact machine-readable verdict helpers
verdict = {
    "flowers_all_tested_widths_positive_mean_gap":
        bool(
            (
                flowers_pair_df[
                    "mean_delta_nmse_200_minus_100"
                ] > 0
            ).all()
        ),

    "data_size_gap_positive_at_all_sizes":
        bool(
            (
                data_pivot[
                    "delta_500_minus_50"
                ] > 0
            ).all()
        ),

    "architecture_gap_positive_all_architectures":
        bool(
            (
                arch_gap[
                    "delta_500_minus_50"
                ] > 0
            ).all()
        ),

    "capacity_gap_positive_all_widths":
        bool(
            (
                cap_gap[
                    "delta_500_minus_50"
                ] > 0
            ).all()
        ),

    "equal_nfe_inversion_found":
        bool(
            "equal_nfe_inv" in globals()
            and len(equal_nfe_inv) > 0
        ),
}

print("\nAUTOMATIC CHECKS")
for k, v in verdict.items():
    print(f"{k}: {v}")

pd.DataFrame(
    [verdict]
).to_csv(
    OUT_DIR / "automatic_verdict.csv",
    index=False
)

print(
    "\nSaved all Step 5 outputs to:",
    OUT_DIR.resolve()
)


STEP 5 SUMMARY

[5A] FLOWERS PAIRED SEED ROBUSTNESS


,width,n_seeds,mean_delta_nmse_200_minus_100,std_delta,fraction_positive,all_positive
0,4,5,0.011206,0.003992,1.0,True
1,16,5,-0.000897,0.004352,0.4,False



[5B] CIFAR DATA-SIZE GAP


C,50,500,delta_500_minus_50
n_train,,,
96,0.329442,0.559254,0.229812
192,0.328217,0.556512,0.228296
384,0.321530,0.551832,0.230302



[5C] CIFAR ARCHITECTURE GAP


C,50,500,delta_500_minus_50
architecture,,,
dilated,0.298916,0.533599,0.234683
local,0.320659,0.552337,0.231678
resnet,0.309351,0.544975,0.235625


... (3 outputs left out)

,C,width,more_accurate_solver,less_accurate_solver,mse_more_accurate,mse_less_accurate,nmse_more_accurate,nmse_less_accurate,nmse_inversion_gap
0,50,8,ddim,deis,0.215740,0.243225,0.363586,0.339413,0.024173
1,50,8,ddim,dpmpp,0.215740,0.243834,0.363586,0.340722,0.022864
2,50,8,ddim,unipc,0.215740,0.243029,0.363586,0.341390,0.022196
3,100,8,ddim,deis,0.196293,0.242124,0.361784,0.340362,0.021422
4,100,8,ddim,dpmpp,0.196293,0.242484,0.361784,0.340815,0.020969
5,100,4,ddim,dpmpp,0.196293,0.242484,0.362139,0.341209,0.020930
6,100,8,ddim,unipc,0.196293,0.242250,0.361784,0.341001,0.020783
7,100,4,ddim,unipc,0.196293,0.242250,0.362139,0.341403,0.020736
8,100,4,ddim,deis,0.196293,0.242124,0.362139,0.342786,0.019353
9,100,1,ddim,deis,0.196293,0.242124,0.411020,0.392354,0.018666



[5F] RAC THRESHOLD SENSITIVITY


C,50,500
rho,,
0.10,NaN,NaN
0.15,NaN,NaN
0.20,NaN,NaN
0.25,NaN,NaN
0.30,NaN,NaN
0.35,1611.0,NaN
0.40,1611.0,NaN
0.45,1611.0,NaN
0.50,1611.0,NaN



AUTOMATIC CHECKS
flowers_all_tested_widths_positive_mean_gap: False
data_size_gap_positive_at_all_sizes: True
architecture_gap_positive_all_architectures: True
capacity_gap_positive_all_widths: True
equal_nfe_inversion_found: True

Saved all Step 5 outputs to: D:\PhD EMP\Paper Writting\STEP_COMPLEXITY\step5_robustness
